# Carga de librerías

In [11]:
# =========================
# 01) LIBRERÍAS Y SETTINGS
# =========================

# Manejo de rutas (más limpio que usar strings)
from pathlib import Path

# Datos
import pandas as pd
import numpy as np
import unicodedata
import re


# Gráficas
import matplotlib.pyplot as plt

# Opcional: para evitar warnings ruidosos (no afecta resultados)
import warnings
warnings.filterwarnings("ignore")

# -------------------------
# Ajustes de visualización
# -------------------------

# Para ver mejor tablas en notebook (puedes ajustar si quieres)
pd.set_option("display.max_columns", 200)   # mostrar muchas columnas
pd.set_option("display.width", 120)         # ancho de impresión en consola/notebook

# Estética base de matplotlib (solo lo básico)
plt.rcParams["figure.dpi"] = 120            # nitidez
plt.rcParams["axes.grid"] = True            # grid por defecto
plt.rcParams["grid.alpha"] = 0.25           # grid suave
plt.rcParams["axes.axisbelow"] = True       # grid por debajo de los puntos/lineas


# Cargar excel

In [12]:
# =========================
# 02) CARGA DEL EXCEL
# =========================

# 1) Ruta al archivo
#    - Si el Excel está en la MISMA carpeta que el notebook, esto basta:
excel_path = Path("/content/Base_CMAT_concentrado_v2.xlsx")

#    - Si está en otra carpeta, pon la ruta completa, por ejemplo:
# excel_path = Path(r"C:\Users\TuUsuario\Documents\concentrado_base_CMAT.xlsx")

# 2) Verificación rápida: que el archivo exista antes de continuar
if not excel_path.exists():
    raise FileNotFoundError(f"No encontré el archivo en: {excel_path.resolve()}")

print("Archivo encontrado:", excel_path.resolve())

# 3) Ver hojas disponibles (así confirmas que siguen siendo años)
xl = pd.ExcelFile(excel_path, engine="openpyxl")
print("Hojas encontradas:", xl.sheet_names)

# 4) Leer todas las hojas y unirlas en un solo DataFrame
dfs = []

for sh in xl.sheet_names:
    # Leer hoja
    df = pd.read_excel(excel_path, sheet_name=sh, engine="openpyxl")

    # Agregar columna Año:
    # - Si el nombre de la hoja es "2024", lo convertimos a int 2024
    # - Si no es numérico, lo dejamos como texto (por seguridad)
    df["Año"] = int(sh) if str(sh).isdigit() else str(sh)

    dfs.append(df)

# Unir todo
data = pd.concat(dfs, ignore_index=True)

# 5) Limpieza ligera de nombres de columnas (quita espacios extra)
data.columns = [c.strip() for c in data.columns]

# 6) Chequeos rápidos (esto te sirve SIEMPRE al inicio)
print("DataFrame final:")
print("Filas:", data.shape[0])
print("Columnas:", data.shape[1])


Archivo encontrado: /content/Base_CMAT_concentrado_v2.xlsx
Hojas encontradas: ['2018', '2019', '2020', '2021', '2022', '2023', '2024', '2025']
DataFrame final:
Filas: 38853
Columnas: 23


In [13]:
# Vista rápida
display(data.head(10))

,ID,Carrera,Materia,Clave materia,Periodo (Prim_Ot),Profesor de materia,Aprobó la materia (Sí/No),Calificación de materia,Estatus,Tomó asesoría (Sí/No),Tema de la asesoría,Profesor de asesoría,Clave profesor de asesoría,Número de asesorías registradas (por materia),Número de asesorías registradas (por materia y tema),"Número de asesorías registradas (por materia, tema y profesor)",Puntuación del diagnóstico 1,Puntuación del diagnóstico 2,Puntuación del diagnóstico 3,Puntuación del diagnóstico 4,Total de puntos,Porcentaje,Año
0,156816,LNI,Cálculo I,MAT1022,OTOÑO,NaN,Sí,8.2,APROBADA,No,NaN,NaN,NaN,0,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,2018
1,156910,LNI,Cálculo I,MAT1022,OTOÑO,NaN,Sí,7.8,APROBADA,No,NaN,NaN,NaN,0,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,2018
2,157150,LCS,Cálculo I,MAT1022,OTOÑO,NaN,Sí,8.7,APROBADA,No,NaN,NaN,NaN,0,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,2018
3,157496,LFP,Cálculo I,MAT1022,OTOÑO,NaN,Sí,7.7,APROBADA,No,NaN,NaN,NaN,0,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,2018
4,157755,LEC,Cálculo I,MAT1022,OTOÑO,NaN,Sí,10,APROBADA,No,NaN,NaN,NaN,0,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,2018
5,158276,LNI,Cálculo I,MAT1022,OTOÑO,NaN,Sí,9,APROBADA,No,NaN,NaN,NaN,0,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,2018
6,158304,LBN,Cálculo I,MAT1022,PRIMAVERA,NaN,Sí,8.6,APROBADA,No,NaN,NaN,NaN,0,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,2018
7,158325,NaN,Cálculo I,MAT1022,PRIMAVERA,NaN,Sí,8.7,APROBADA,No,NaN,NaN,NaN,0,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,2018
8,158374,NaN,Cálculo I,MAT1022,PRIMAVERA,NaN,Sí,9.2,APROBADA,No,NaN,NaN,NaN,0,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,2018
9,158454,LNM,Cálculo I,MAT1022,PRIMAVERA,NaN,Sí,9.7,APROBADA,No,NaN,NaN,NaN,0,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,2018


In [ ]:
# ==========================================================
# 03) SELECCIÓN DE COLUMNAS + LIMPIEZA + TIPOS + CHECKS
# ==========================================================
#
# (1) Quedarnos SOLO con las columnas que usaremos en las gráficas
# (2) Limpiar texto (espacios, mayúsculas) sin romper NA
# (3) Convertir tipos (asesorías, calificación, año)
# (4) Hacer validaciones rápidas (missing, uniques, rangos, duplicados)
#

# ----------------------------------------------------------
# 1) Definimos columnas requeridas (las únicas que nos importan)
# ----------------------------------------------------------
cols_keep = [
    "ID",
    "Carrera",
    "Materia",
    "Periodo (Prim_Ot)",
    "Número de asesorías registradas (por materia)",
    "Estatus",
    "Calificación de materia",
    "Año",
]

# ----------------------------------------------------------
# 2) Verificamos que TODAS las columnas existan
#    (si cambian el nombre del encabezado en el Excel, aquí lo detectas)
# ----------------------------------------------------------
missing_cols = [c for c in cols_keep if c not in data.columns]
if missing_cols:
    raise KeyError(
        "Faltan estas columnas en el Excel (revisa nombres exactos de encabezados):\n"
        + "\n".join(missing_cols)
    )

# ----------------------------------------------------------
# 3) Reducimos el DataFrame a solo estas columnas
#    (esto baja tamaño y te evita cargar cosas que no usarás)
# ----------------------------------------------------------
df = data[cols_keep].copy()

# ----------------------------------------------------------
# 4) Limpieza de columnas de texto
#    Importante: usamos dtype "string" de pandas para conservar NA.
#    (Si haces astype(str) pandas convierte NaN en el texto "nan" y eso ensucia filtros)
# ----------------------------------------------------------
text_cols = ["ID", "Carrera", "Materia", "Periodo (Prim_Ot)", "Estatus"]

for c in text_cols:
    # Convertimos a "string" (mantiene NA) y quitamos espacios alrededor
    df[c] = df[c].astype("string").str.strip()

# Estandarización (opcional pero MUY útil):
# - evita que existan categorías duplicadas por mayúsculas/minúsculas
#   ejemplo: "Otoño" vs "OTOÑO" vs "otoño"
df["Carrera"] = df["Carrera"].str.upper()
df["Materia"] = df["Materia"].str.upper()
df["Periodo (Prim_Ot)"] = df["Periodo (Prim_Ot)"].str.upper()
df["Estatus"] = df["Estatus"].str.upper()

# ----------------------------------------------------------
# 5) Conversión a numéricos
# ----------------------------------------------------------

# 5.1) Número de asesorías
# - pd.to_numeric con errors="coerce" convierte valores raros a NaN
# - luego lo pasamos a Int64 (entero que permite NA)
df["Número de asesorías registradas (por materia)"] = pd.to_numeric(
    df["Número de asesorías registradas (por materia)"],
    errors="coerce"
).astype("Int64")

# 5.2) Calificación
# - la dejamos como float (puede tener decimales)
df["Calificación de materia"] = pd.to_numeric(
    df["Calificación de materia"],
    errors="coerce"
)

# 5.3) Año
df["Año"] = pd.to_numeric(df["Año"], errors="coerce").astype("Int64")

# ==========================================================
# 6) CHECKS RÁPIDOS
# ==========================================================

print("df listo con columnas reducidas")
print("Filas:", df.shape[0], "| Columnas:", df.shape[1])
display(df.head(5))

# 6.1) Tipos
print("\n--- Dtypes ---")
print(df.dtypes)

# 6.2) Resumen por columna: missing + número de valores únicos
# - missing: cuántos NaN hay por columna
# - n_unique: cuántos valores distintos hay (incluyendo NA)
resumen = pd.DataFrame({
    "dtype": df.dtypes.astype(str),
    "missing": df.isna().sum(),
    "n_unique": df.nunique(dropna=False),
}).sort_index()

print("\n--- Resumen (missing / uniques) ---")
display(resumen)

# 6.3) Uniques clave (útiles para validar rápidamente)
# - Año: qué años existen
# - Periodo: qué categorías existen (por ejemplo OTOÑO, PRIMAVERA)
# - Estatus: APROBADA, REPROBADA, BAJA VOLUNTARIA, etc.
print("\n--- Valores únicos clave ---")
print("Materia:", sorted(df["Materia"].dropna().unique().tolist()))
print("Año:", sorted(df["Año"].dropna().unique().tolist()))
print("Periodo (Prim_Ot):", sorted(df["Periodo (Prim_Ot)"].dropna().unique().tolist()))
print("Estatus:", sorted(df["Estatus"].dropna().unique().tolist()))

# 6.4) Distribuciones (para ver si hay categorías raras)
print("\n--- Distribución rápida de categorías ---")
display(df["Periodo (Prim_Ot)"].value_counts(dropna=False))
display(df["Estatus"].value_counts(dropna=False))

# 6.5) Validaciones sencillas de rango (no detiene nada, solo avisa)
# - Calificación normalmente debe estar entre 0 y 10
# - Asesorías no deberían ser negativas
bad_grade = df["Calificación de materia"].notna() & (
    (df["Calificación de materia"] < 0) | (df["Calificación de materia"] > 10)
)
bad_ases = df["Número de asesorías registradas (por materia)"].notna() & (
    df["Número de asesorías registradas (por materia)"] < 0
)

print("\n--- Validaciones de rango ---")
print("Calificaciones fuera de [0, 10]:", int(bad_grade.sum()))
print("Asesorías negativas:", int(bad_ases.sum()))

# 6.6) Duplicados "lógicos"
# Esto NO necesariamente es error: puede haber varias filas por alumno/materia/año.
# Pero es importante detectarlo porque para las gráficas quizá agregaremos (sum/mean/max).
dup_key = ["ID", "Carrera", "Materia", "Año"]
dup_count = df.duplicated(subset=dup_key).sum()

print(f"\n--- Duplicados por llave {dup_key} ---")
print("Duplicados:", dup_count)

# Mostramos ejemplos si existen
if dup_count > 0:
    display(
        df[df.duplicated(subset=dup_key, keep=False)]
        .sort_values(dup_key)
        .head(10)
    )


df listo con columnas reducidas
Filas: 38853 | Columnas: 8


,ID,Carrera,Materia,Periodo (Prim_Ot),Número de asesorías registradas (por materia),Estatus,Calificación de materia,Año
0,156816,LNI,CÁLCULO I,OTOÑO,0,APROBADA,8.2,2018
1,156910,LNI,CÁLCULO I,OTOÑO,0,APROBADA,7.8,2018
2,157150,LCS,CÁLCULO I,OTOÑO,0,APROBADA,8.7,2018
3,157496,LFP,CÁLCULO I,OTOÑO,0,APROBADA,7.7,2018
4,157755,LEC,CÁLCULO I,OTOÑO,0,APROBADA,10.0,2018



--- Dtypes ---
ID                                               string[python]
Carrera                                          string[python]
Materia                                          string[python]
Periodo (Prim_Ot)                                string[python]
Número de asesorías registradas (por materia)             Int64
Estatus                                          string[python]
Calificación de materia                                 float64
Año                                                       Int64
dtype: object

--- Resumen (missing / uniques) ---


,dtype,missing,n_unique
Año,Int64,0,8
Calificación de materia,float64,4665,102
Carrera,string,1628,100
Estatus,string,0,8
ID,string,0,11608
Materia,string,552,11
Número de asesorías registradas (por materia),Int64,0,31
Periodo (Prim_Ot),string,0,3



--- Valores únicos clave ---
Materia: ['CÁLCULO I', 'CÁLCULO II', 'ECUACIONES DIFERENCIALES ORDINARIAS', 'ESTADÍSTICA INFERENCIAL', 'ESTADÍSTICA PARA CIENCIAS DE LA SALUD', 'ESTADÍSTICA PARA CIENCIAS SOCIALES', 'ESTADÍSTICA PARA NEGOCIOS', 'MATEMÁTICAS UNIVERSITARIAS', 'TEORÍA DE MATRICES', 'ÁLGEBRA LINEAL']
Año: [2018, 2019, 2020, 2021, 2022, 2023, 2024, 2025]
Periodo (Prim_Ot): ['OTOÑO', 'PRIMAVERA', 'VERANO']
Estatus: ['ACREDITADA', 'APROBADA', 'BAJA ACADEMICA', 'BAJA VOLUNTARIA', 'EQUIVALENCIA', 'REPROBADA', 'RETIRO', 'REVALIDADA']

--- Distribución rápida de categorías ---


,count
Periodo (Prim_Ot),
OTOÑO,20047
PRIMAVERA,17438
VERANO,1368


,count
Estatus,
APROBADA,28120
REPROBADA,6612
BAJA VOLUNTARIA,2787
RETIRO,1114
EQUIVALENCIA,130
BAJA ACADEMICA,70
REVALIDADA,16
ACREDITADA,4



--- Validaciones de rango ---
Calificaciones fuera de [0, 10]: 0
Asesorías negativas: 0

--- Duplicados por llave ['ID', 'Carrera', 'Materia', 'Año'] ---
Duplicados: 6306


,ID,Carrera,Materia,Periodo (Prim_Ot),Número de asesorías registradas (por materia),Estatus,Calificación de materia,Año
27048,140079,LNA,CÁLCULO I,OTOÑO,26,APROBADA,9.2,2023
27049,140079,LNA,CÁLCULO I,OTOÑO,26,APROBADA,9.2,2023
27050,140079,LNA,CÁLCULO I,OTOÑO,26,APROBADA,9.2,2023
27051,140079,LNA,CÁLCULO I,OTOÑO,26,APROBADA,9.2,2023
27052,140079,LNA,CÁLCULO I,OTOÑO,26,APROBADA,9.2,2023
27053,140079,LNA,CÁLCULO I,OTOÑO,26,APROBADA,9.2,2023
27054,140079,LNA,CÁLCULO I,OTOÑO,26,APROBADA,9.2,2023
27055,140079,LNA,CÁLCULO I,OTOÑO,26,APROBADA,9.2,2023
27056,140079,LNA,CÁLCULO I,OTOÑO,26,APROBADA,9.2,2023
27057,140079,LNA,CÁLCULO I,OTOÑO,26,APROBADA,9.2,2023


In [15]:
# ==========================================================
# 04) FILTRO DE ESTATUS (QUITAR BAJA VOLUNTARIA Y RETIRO)
# ==========================================================

# 1) Definir lista de estatus a excluir
estatus_excluir = ["BAJA VOLUNTARIA", "RETIRO"]

# 2) Crear tabla filtrada
#    - keep: nos quedamos con todo lo que NO esté en estatus_excluir
df_ok = df[~df["Estatus"].isin(estatus_excluir)].copy()

# 3) (Opcional, pero recomendado) resetear índice para que quede limpio
df_ok = df_ok.reset_index(drop=True)

# ==========================================================
# CHECKS RÁPIDOS
# ==========================================================

print("Tabla filtrada creada: df_ok")
print("Filas antes:", df.shape[0])
print("Filas después:", df_ok.shape[0])
print("Filas eliminadas:", df.shape[0] - df_ok.shape[0])

print("\n--- Distribución de Estatus (ANTES) ---")
display(df["Estatus"].value_counts(dropna=False))

print("\n--- Distribución de Estatus (DESPUÉS) ---")
display(df_ok["Estatus"].value_counts(dropna=False))

Tabla filtrada creada: df_ok
Filas antes: 38853
Filas después: 34952
Filas eliminadas: 3901

--- Distribución de Estatus (ANTES) ---


,count
Estatus,
APROBADA,28120
REPROBADA,6612
BAJA VOLUNTARIA,2787
RETIRO,1114
EQUIVALENCIA,130
BAJA ACADEMICA,70
REVALIDADA,16
ACREDITADA,4



--- Distribución de Estatus (DESPUÉS) ---


,count
Estatus,
APROBADA,28120
REPROBADA,6612
EQUIVALENCIA,130
BAJA ACADEMICA,70
REVALIDADA,16
ACREDITADA,4


In [ ]:
# ==========================================================
# 05) FORMATO DE TEXTO: "Título" (Primera mayúscula por palabra)
# ==========================================================

df_fmt = df_ok.copy()

# 1) Crear columnas "key" (estables) para agrupar/filtrar sin depender de formato bonito
df_fmt["Carrera_key"] = df_fmt["Carrera"].astype("string").str.upper().str.strip()
df_fmt["Materia_key"] = df_fmt["Materia"].astype("string").str.upper().str.strip()
df_fmt["Periodo_key"] = df_fmt["Periodo (Prim_Ot)"].astype("string").str.upper().str.strip()
df_fmt["Estatus_key"] = df_fmt["Estatus"].astype("string").str.upper().str.strip()

# 2) Función sencilla: Title Case respetando acentos (Python maneja unicode bien)
#    - title(): pone Primera Mayúscula Por Palabra, el resto minúscula
#    - Conserva letras con acento (ÁÉÍÓÚÑ) correctamente en general
def to_title(s: pd.Series) -> pd.Series:
    return s.astype("string").str.strip().str.lower().str.title()

# 3) Aplicar formato "bonito" a columnas de texto principales
df_fmt["Carrera"] = to_title(df_fmt["Carrera"])
df_fmt["Materia"] = to_title(df_fmt["Materia"])


# Regex de números romanos comunes (puedes ampliarlo si quieres)
roman_pattern = re.compile(r"\b(i|ii|iii|iv|v|vi|vii|viii|ix|x|xi|xii|xiii|xiv|xv)\b", flags=re.IGNORECASE)

def fix_roman_numerals(text: str) -> str:
    """
    Convierte números romanos escritos como 'Ii' o 'Iv' a 'II' o 'IV'
    sin tocar acentos del resto del texto.
    """
    if text is None:
        return text
    text = str(text)

    # Reemplaza cada match por su versión en MAYÚSCULAS
    return roman_pattern.sub(lambda m: m.group(0).upper(), text)

# Aplicar solo a Materia (la que trae I, II, III, etc.)
df_fmt["Materia"] = df_fmt["Materia"].astype("string").apply(fix_roman_numerals)
df_fmt["Periodo (Prim_Ot)"] = to_title(df_fmt["Periodo (Prim_Ot)"])
df_fmt["Estatus"] = to_title(df_fmt["Estatus"])

# 4) Checks rápidos
print("✅ df_fmt listo: texto en formato Título + llaves *_key en MAYÚSCULAS")
display(df_fmt[["Carrera", "Carrera_key", "Materia", "Materia_key", "Periodo (Prim_Ot)", "Periodo_key", "Estatus", "Estatus_key"]].head(8))

print("\n--- Uniques de ejemplo (bonito) ---")
print("Periodo:", sorted(df_fmt["Periodo (Prim_Ot)"].dropna().unique().tolist()))
print("Estatus:", sorted(df_fmt["Estatus"].dropna().unique().tolist()))


✅ df_fmt listo: texto en formato Título + llaves *_key en MAYÚSCULAS


,Carrera,Carrera_key,Materia,Materia_key,Periodo (Prim_Ot),Periodo_key,Estatus,Estatus_key
0,Lni,LNI,Cálculo I,CÁLCULO I,Otoño,OTOÑO,Aprobada,APROBADA
1,Lni,LNI,Cálculo I,CÁLCULO I,Otoño,OTOÑO,Aprobada,APROBADA
2,Lcs,LCS,Cálculo I,CÁLCULO I,Otoño,OTOÑO,Aprobada,APROBADA
3,Lfp,LFP,Cálculo I,CÁLCULO I,Otoño,OTOÑO,Aprobada,APROBADA
4,Lec,LEC,Cálculo I,CÁLCULO I,Otoño,OTOÑO,Aprobada,APROBADA
5,Lni,LNI,Cálculo I,CÁLCULO I,Otoño,OTOÑO,Aprobada,APROBADA
6,Lbn,LBN,Cálculo I,CÁLCULO I,Primavera,PRIMAVERA,Aprobada,APROBADA
7,<NA>,<NA>,Cálculo I,CÁLCULO I,Primavera,PRIMAVERA,Aprobada,APROBADA



--- Uniques de ejemplo (bonito) ---
Periodo: ['Otoño', 'Primavera', 'Verano']
Estatus: ['Acreditada', 'Aprobada', 'Baja Academica', 'Equivalencia', 'Reprobada', 'Revalidada']


# Gráficos de correlación

In [17]:
# ==========================================================
# 06) GRÁFICAS DE CORRELACIÓN (ASESORÍAS VS CALIFICACIÓN)
#     - Guarda PNGs en: "graficos CMAT/<AÑO>/"
#     - Por cada AÑO, por cada PERIODO, por cada MATERIA
#     - Caso especial: "Matemáticas Universitarias" lleva línea vertical
# ==========================================================


# ----------------------------------------------------------
# 0) PARÁMETROS
# ----------------------------------------------------------
calif_min_aprobatoria = 7.5     # línea horizontal (verde punteada)
min_asesorias_mu = 3            # línea vertical (verde sólida) SOLO para Matemáticas Universitarias

# Carpeta base donde se guardará todo
# - si quieres guardarlo en otra parte, cambia output_root:
output_root = Path("/content/drive/MyDrive/CMAT/")  # carpeta actual del notebook
output_dir = output_root / "graficos CMAT"

# Crear carpeta base si no existe
output_dir.mkdir(parents=True, exist_ok=True)
print("Carpeta base de salida:", output_dir.resolve())

# ----------------------------------------------------------
# 1) Helpers sencillos
# ----------------------------------------------------------

def quitar_acentos(txt: str) -> str:
    """Quita acentos para comparar textos de forma robusta."""
    if txt is None:
        return ""
    txt = str(txt)
    return "".join(
        c for c in unicodedata.normalize("NFD", txt)
        if unicodedata.category(c) != "Mn"
    )

def safe_filename(name: str) -> str:
    """
    Convierte un texto a un nombre de archivo seguro.
    """
    bad = '<>:"/\\|?*'
    name = str(name)
    for ch in bad:
        name = name.replace(ch, "")
    name = name.strip().replace("  ", " ").replace(" ", "_")
    return name

# ----------------------------------------------------------
# 2) Construir tabla "para graficar"
# ----------------------------------------------------------
# Regla simple:
# - asesorías: SUMA (si hay varias filas del mismo alumno en el mismo grupo)
# - calificación: MAX (te quedas con la mejor/definitiva en ese grupo)
#
# Grupo = alumno + año + periodo + materia

# ==========================================================
# 2) Construir tabla "para graficar" SIN mezclar años/periodos
#    y (recomendado) SIN mezclar carreras
# ==========================================================

df_plot = (
    df_fmt
    .groupby(["ID", "Carrera_key", "Año", "Materia_key"], as_index=False)
    .agg({
        # IMPORTANTE:
        # Si el Excel ya trae el total de asesorías por alumno/materia/periodo,
        # NO se debe sumar. Usamos MAX para evitar inflar por duplicados.
        "Número de asesorías registradas (por materia)": "max",

        # Para calificación, MAX suele funcionar si hay registros repetidos.
        # (si fuera "última calificación", habría que usar fecha — pero aquí no hay.)
        "Calificación de materia": "max",

        # Texto bonito para títulos
        "Carrera": "first",
        "Materia": "first",
    })
    .rename(columns={
        "Número de asesorías registradas (por materia)": "Asesorias",
        "Calificación de materia": "Calificacion",
    })
)

print("df_plot listo para gráficas")
print("Filas:", df_plot.shape[0], "| Columnas:", df_plot.shape[1])
display(df_plot.head(5))

# ----------------------------------------------------------
# 3) Loop para generar TODAS las gráficas y guardarlas
#    - Por AÑO y por MATERIA (sin periodo)
# ----------------------------------------------------------

# Lista de años disponibles
years = sorted(df_plot["Año"].dropna().unique().tolist())

for year in years:
    # Crear carpeta por año: graficos CMAT/2023/
    year_dir = output_dir / str(int(year))
    year_dir.mkdir(parents=True, exist_ok=True)

    # Filtrar datos de ese año
    df_y = df_plot[df_plot["Año"] == year].copy()

    # Materias dentro del año
    materias = sorted(df_y["Materia"].dropna().unique().tolist())

    for materia in materias:
        df_m = df_y[df_y["Materia"] == materia].copy()

        # Quitar filas sin calificación o sin asesorías (opcional)
        df_m = df_m[df_m["Calificacion"].notna() & df_m["Asesorias"].notna()].copy()

        n = df_m.shape[0]
        if n == 0:
            continue

        # ----------------------------------------------------------
        # Detectar si la materia es "Matemáticas Universitarias"
        # ----------------------------------------------------------
        materia_norm = quitar_acentos(materia).upper()
        es_mate_uni = (materia_norm == "MATEMATICAS UNIVERSITARIAS")

        # ----------------------------------------------------------
        # Crear figura y scatter (naranja)
        # ----------------------------------------------------------
        plt.figure(figsize=(12.5, 5.8))

        plt.scatter(
            df_m["Asesorias"],
            df_m["Calificacion"],
            color="orange",
            alpha=0.85,
            s=45
        )

        # Línea horizontal: calificación mínima aprobatoria
        plt.axhline(
            calif_min_aprobatoria,
            linestyle="--",
            linewidth=2,
            color="darkgreen",
            label="Calificación mínima aprobatoria"
        )

        # Línea vertical SOLO si es Matemáticas Universitarias
        if es_mate_uni:
            plt.axvline(
                min_asesorias_mu,
                linestyle="-",
                linewidth=2,
                color="darkgreen",
                label="Mínimo de asesorías"
            )

        # ----------------------------------------------------------
        # Título y ejes (sin periodo)
        # ----------------------------------------------------------
        title = f"Asesorías vs Calificación — {materia} {int(year)}."
        plt.title(title, fontsize=14)

        plt.xlabel("Asesorías", fontsize=12)
        plt.ylabel("Calificación (numérica)", fontsize=12)

        plt.ylim(-0.2, 10.2)
        plt.xlim(-0.5, float(df_m["Asesorias"].max()) + 1.5)

        plt.legend(loc=("lower right" if es_mate_uni else "upper right"))
        plt.tight_layout()

        # ----------------------------------------------------------
        # Guardar en carpeta del año (sin periodo)
        # ----------------------------------------------------------
        file_name = safe_filename(f"{materia}_{int(year)}") + ".png"
        save_path = year_dir / file_name

        plt.savefig(save_path, dpi=200)
        plt.close()

print("Listo: gráficas generadas y guardadas por año en:", output_dir.resolve())

Carpeta base de salida: /content/drive/MyDrive/CMAT/graficos CMAT
df_plot listo para gráficas
Filas: 28635 | Columnas: 8


,ID,Carrera_key,Año,Materia_key,Asesorias,Calificacion,Carrera,Materia
0,111083,LFP,2024,CÁLCULO I,0,10.0,Lfp,Cálculo I
1,134718,LIS,2021,CÁLCULO I,0,NaN,Lis,Cálculo I
2,134718,LIS,2021,CÁLCULO II,0,NaN,Lis,Cálculo II
3,134718,LIS,2021,MATEMÁTICAS UNIVERSITARIAS,0,NaN,Lis,Matemáticas Universitarias
4,134718,LIS,2021,ÁLGEBRA LINEAL,0,NaN,Lis,Álgebra Lineal


Listo: gráficas generadas y guardadas por año en: /content/drive/MyDrive/CMAT/graficos CMAT


# Gráfico de Lineas

In [18]:
# ==========================================================
# 07) GRÁFICAS DE LÍNEAS
#     Promedio de calificación por año:
#       - Con asesoría (>=1)
#       - Sin asesoría (=0)
#     Se guarda 1 gráfico por materia en:
#       "graficos de lineas/"
# ==========================================================

# ----------------------------------------------------------
# 0) PARÁMETROS (editables)
# ----------------------------------------------------------
year_min = 2020
year_max = 2024

calif_min_aprobatoria = 7.5  # línea horizontal gris (como en tu ejemplo)

# Carpeta de salida
output_root = Path("/content/drive/MyDrive/CMAT/")
line_dir = output_root / "graficos de lineas"
line_dir.mkdir(parents=True, exist_ok=True)
print("✅ Carpeta de salida:", line_dir.resolve())

# ----------------------------------------------------------
# 1) Partimos de df_plot (del Módulo 6)
#    df_plot tiene columnas: Asesorias, Calificacion, Materia, Periodo, Año, etc.
#
#    Si NO tienes df_plot en memoria por cualquier razón, asegúrate de correr
#    el Módulo 6 (la parte donde se construye df_plot).
# ----------------------------------------------------------

# Filtrar años de interés
df_lines = df_plot[
    df_plot["Año"].between(year_min, year_max)
].copy()

# Nos quedamos con filas con calificación válida
df_lines = df_lines[df_lines["Calificacion"].notna()].copy()

# Crear etiqueta de grupo:
# - "Con asesoría" si Asesorias >= 1
# - "Sin asesoría" si Asesorias == 0
df_lines["Grupo_Asesoria"] = np.where(df_lines["Asesorias"] >= 1, "Con asesoría", "Sin asesoría")

# ----------------------------------------------------------
# 2) Agregar por Materia y Año
#    Queremos promedio de calificación por año para cada grupo.
# ----------------------------------------------------------
summary = (
    df_lines
    .groupby(["Materia", "Año", "Grupo_Asesoria"], as_index=False)
    .agg(
        Calificacion_promedio=("Calificacion", "mean"),
        n=("Calificacion", "count")  # para saber el tamaño de muestra por punto
    )
)

# ----------------------------------------------------------
# 3) Función simple para guardar gráficos por materia
# ----------------------------------------------------------
def safe_filename(name: str) -> str:
    """Nombre de archivo seguro (sin caracteres inválidos para Windows)."""
    bad = '<>:"/\\|?*'
    name = str(name)
    for ch in bad:
        name = name.replace(ch, "")
    name = name.strip().replace("  ", " ").replace(" ", "_")
    return name

# Años en el rango (para que el eje X sea consistente)
years = list(range(year_min, year_max + 1))

materias = sorted(summary["Materia"].dropna().unique().tolist())

for materia in materias:
    tmp = summary[summary["Materia"] == materia].copy()

    # Crear tablas separadas por grupo para graficar líneas limpias
    tmp_con = tmp[tmp["Grupo_Asesoria"] == "Con asesoría"].set_index("Año").reindex(years)
    tmp_sin = tmp[tmp["Grupo_Asesoria"] == "Sin asesoría"].set_index("Año").reindex(years)

    # Valores a graficar
    y_con = tmp_con["Calificacion_promedio"].values
    y_sin = tmp_sin["Calificacion_promedio"].values

    # Si para esa materia no hay datos en absoluto, skip
    if np.all(np.isnan(y_con)) and np.all(np.isnan(y_sin)):
        continue

    # ------------------------------------------------------
    # Plot
    # ------------------------------------------------------
    plt.figure(figsize=(11.5, 5.5))

    # Con asesoría (naranja)
    plt.plot(
        years, y_con,
        marker="o",
        linewidth=3,
        linestyle="-",
        color="orange",
        label="Con asesoría"
    )

    # Sin asesoría (verde, línea punteada)
    plt.plot(
        years, y_sin,
        marker="o",
        linewidth=3,
        linestyle="--",
        color="darkgreen",
        label="Sin asesoría"
    )

    # Línea horizontal de calificación mínima aprobatoria (gris punteada)
    plt.axhline(calif_min_aprobatoria, linestyle="--", linewidth=2, color="gray", alpha=0.8)

    # Título (2 líneas como tu ejemplo)
    # Nota: ponemos (year_min–year_max) para que siempre se vea el rango real
    plt.title(
        f"Promedio de calificación por año — {materia} ({year_min}–{year_max}).\n"
        f"Comparación con asesorías/sin asesorías.",
        fontsize=14
    )

    plt.xlabel("Año", fontsize=12)
    plt.ylabel("Calificación promedio", fontsize=12)

    plt.xticks(years)
    plt.ylim(6.5, 10.2)
    plt.yticks(np.arange(6.5, 10.5, 0.5))

    plt.legend(loc="lower right")
    plt.tight_layout()

    # Guardar
    file_name = safe_filename(f"Promedio_{materia}_{year_min}_{year_max}.png")
    save_path = line_dir / file_name

    plt.savefig(save_path, dpi=200)
    plt.close()

print("✅ Listo: gráficas de líneas guardadas en:", line_dir.resolve())

✅ Carpeta de salida: /content/drive/MyDrive/CMAT/graficos de lineas
✅ Listo: gráficas de líneas guardadas en: /content/drive/MyDrive/CMAT/graficos de lineas
